##**[11주차]실습**
- 교재를 참고하여 아래의 실습을 수행하시오.
- 모든 코드의 결과를 출력하여, .ipynb의 링크를 **[11주차]/[11주차]과제**에 제출하시오.\
(실습 제출 예시: 11주차_2020XXXX_이름.ipynb 코드 링크)

In [27]:
print("2353881, 최대영")

2353881, 최대영


In [28]:
# 설치 후 세션 재시작, 재시작 후에는 해당 셀 스킵
!pip install numpy==1.24.1 regex==2017.4.5 requests==2.27.1 tqdm==4.64.0 fire==0.5.0

  Using cached numpy-1.24.1.tar.gz (10.9 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [29]:
from google.colab import auth

auth.authenticate_user()
!gcloud config get-value account

tigers7101@gmail.com


In [30]:
# google drive 연결
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
import sys

# Add the directory containing utils.py to sys.path
sys.path.append('/content/sample_data/deep')

In [32]:
!pip install --upgrade regex

###**예제 1) picoGPT의 decoder를 작성해본다.**
- **[실습목표]** picoGPT의 decode를 작성하며, transformer의 구조를 파악해본다.


In [33]:
import numpy as np
from tqdm import tqdm
from utils import load_encoder_hparams_and_params

#### **1) main() 메소드**
- GPT-2모델을 사용하여 텍스트 생성을 실행할 때 필요한 입력값을 설정
- 아래의 각 매개변수의 역할을 서술하시오.
  - prompt(str): 모델에게 줄 초기 입력 문장(시드 텍스트)입니다
  - n_tokens_to_generate(int): 모델이 새롭게 생성할 토큰(단어 조각)의 개수를 지정합니다

In [34]:
def main(prompt: str, n_tokens_to_generate: int = 40, model_size: str = "124M", models_dir: str = "models"):
    encoder, hparams, params = load_encoder_hparams_and_params(model_size, models_dir)
    input_ids = encoder.encode(prompt)
    assert len(input_ids) + n_tokens_to_generate < hparams["n_ctx"]
    output_ids = generate(input_ids, params, hparams["n_head"], n_tokens_to_generate)
    output_text = encoder.decode(output_ids)
    return output_text

#### **2) generate() 메소드**
- 주어진 입력 시퀀스에 대해 추가적인 토큰을 생성하는 함수
- generate()함수 내에서 gpt2 함수를 통해 입력 시퀀스에 대한 예측을 수행
- 아래의 각 변수의 역할에 대해 서술
  - logits: 델이 "다음에 올 가능성이 얼마나 높은가"를 수치화하는 역할


In [35]:
def generate(inputs, params, n_head, n_tokens_to_generate):
    for _ in tqdm(range(n_tokens_to_generate), "generating"):
        logits = gpt2(inputs, **params, n_head=n_head) # 1. gpt2 함수를 호출하여 현재 입력 시퀀스에 대한 로짓(예측값) 계산
        next_id = np.argmax(logits[-1]) # 2. 마지막 타임스텝(-1)의 로짓에서 가장 값이 큰 인덱스(가장 확률 높은 단어)를 추출
        inputs.append(int(next_id)) # 3. 추출된 다음 토큰 ID를 입력 시퀀스에 추가 (다음 반복의 입력이 됨)
    return inputs # 모든 생성이 완료된 최종 시퀀스 반환

#### **3) gpt2() 메소드**
- 입력 시퀀스에 대해 예측을 수행

In [36]:
def gpt2(inputs, wte, wpe, blocks, ln_f, n_head):
    x = wte[inputs] + wpe[range(len(inputs))] # 1. 입력 토큰을 임베딩 벡터로 변환 (Token Embedding + Positional Embedding)
    # wte: 단어 그 자체의 의미, wpe: 단어의 위치 정보
    for block in blocks:
        x = transformer_block(x, **block, n_head=n_head) # 2. 트랜스포머 블록(Transformer Blocks)을 순차적으로 통과
    # 여러 개의 층을 거치며 문맥 정보를 심화 학습합니다.
    return layer_norm(x, **ln_f) @ wte.T # 3. 최종 레이어 노멀라이제이션 및 로짓(logits) 계산

#### **4) transformer_block()메소드**
- 멀티헤드 어텐션(Multi Head Attention, MHA)와 피드-포워드 신경망(Feed Foward Network)를 포함하는 트렌스포머 블록을 구현한 함수



In [37]:
def transformer_block(x, mlp, attn, ln_1, ln_2, n_head):
    x = x + mha(layer_norm(x, **ln_1), **attn, n_head=n_head) # 입력 x를 정규화(ln_1)한 뒤 어텐션을 수행하고, 그 결과를 원래의 x와 더합니다.
    x = x + ffn(layer_norm(x, **ln_2), **mlp)
    # 2. Feed-Forward Network (MLP) 단계 + 잔차 연결(Residual Connection)
    return x

#### **5) mha()**
- 멀티헤드 어텐션함수는 입력 벡터를 여러 개의 어텐션 헤드로 나누어 병렬로 어텐션을 계산한 후, 그 결과를 결합하는 역할
- 입력 벡터를 여러개의 어텐션 헤드로 나누는 이유는 무엇인가?
- : 헤드를 여러 개 두면 각각의 헤드가 이러한 서로 다른 관계들을 각자의 영역에서 전문적으로 포착할 수 있습니다.


In [38]:
def mha(x, c_attn, c_proj, n_head):
    x = linear(x, **c_attn)
    qkv_heads = list(map(lambda x: np.split(x, n_head, axis=-1), np.split(x, 3, axis=-1)))
    causal_mask = (1 - np.tri(x.shape[0], dtype=x.dtype)) * -1e10
    out_heads = [attention(q, k, v, causal_mask) for q, k, v in zip(*qkv_heads)]
    x = linear(np.hstack(out_heads), **c_proj)
    return x

#### **6) attention() 메소드**
- 어텐션 함수는 주어진 쿼리(Q), 키(K), 값(V) 및 마스크를 사용하여 어텐션 가중치를 계산하는 함수이다.


In [39]:
def attention(q, k, v, mask):
    return softmax(q @ k.T / np.sqrt(q.shape[-1]) + mask) @ v # Softmax를 취해 어텐션 가중치를 구하고 값(V)을 곱합니다.

#### **7) ffn() 메소드**
- 피드-포워드 신경망 함수는 트렌스포머 모델에서 사용되는 구성 요소 중 하나로, 입력 벡터에 대해 두 번의 선형 변환과 활성화 함수를 적용하여 출력 벡터를 생성하는 함수이다.
- 멀티헤드 어텐션 이후에 적용.

In [40]:
def ffn(x, c_fc, c_proj): # (차원 축소)
    x = gelu(linear(x, **c_fc))
    return linear(x, **c_proj)

#### **8, 9, 10) linear(), layer_norm(), gelu(), softmax() 메소드**
- 각각 선형 변환 함수, 레이어 정규화 함수, GELU 활성화 함수, Softmax 함수를 의미하며, 각 함수의 역할은 아래의 설명과 같다.
- **linear()**: 입력 벡터에 대해 선형 변환을 수행.
- **layer_norm()**: 각 레이어의 출력에 정규화를 수행하여 학습 안정성을 높이고, 수렴 속도를 향상.
- **gelu()**: 입력 값을 비선형적으로 변환하여 모델의 표현력을 향상
- **softmax()**: 입력 벡터를 확률 분포로 변환하는 함수, 출력 벡터의 요소들을 0과 1 사이의 값으로 매핑 시키며, 전체 합이 1이 되도록 한다.

In [41]:
def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def layer_norm(x, g, b, eps: float = 1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    variance = np.var(x, axis=-1, keepdims=True)
    return g * (x - mean) / np.sqrt(variance + eps) + b

def linear(x, w, b):
    return x @ w + b

#### **11) prompt 입력 후 결과 출력**


In [43]:
if __name__ == "__main__":
    print(main(prompt="다이어트 보충제 추천해줘."))
    # prompt를 여러가지로 변경하며, 테스트 해보기(선택사항)

generating: 100%|██████████| 40/40 [01:01<00:00,  1.55s/it]

다이어트 보충제 추천해줘.

아마이 이어트 보충제 추천해줘.
